# Research Question 3: Indoor and Outdoor Participation Forecasting

This executable notebook uses three expanding-window rolling-origin validation folds, selects models using mean MAE and Total Variation, holds 2022/23 out for final testing, applies additive log-ratio modelling, runs parameter sensitivity analyses, and forecasts every survey wave from 2023/24 to 2028/29. All implementation comments are in English.


In [1]:
from __future__ import annotations

import argparse
import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any

os.environ.setdefault("MPLCONFIGDIR", "/tmp/q3_matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy.stats import chi2


In [2]:
# =============================================================================
# STEP 1 — Define the fixed settings used throughout the analysis
# =============================================================================
# Keeping these values in one place makes the whole workflow reproducible.
# RANDOM_STATE fixes the random-number sequence used by tree-based models.
# ALR_EPS prevents log(0) when the four outcome proportions are transformed.
# The primary analysis uses 10^-6, as requested.  A formal sensitivity analysis
# repeats evaluation at 10^-4 and 10^-5 and writes the comparison to CSV.
RANDOM_STATE = 42
ALR_EPS = 1e-6
ALR_EPS_SENSITIVITY = (1e-4, 1e-5, 1e-6)
ANNUAL_POOLING_STRENGTH = 5.0
MONTHLY_POOLING_STRENGTH = 20.0

# Expanding-window rolling-origin folds.  Labels use actual survey waves rather
# than ordinal phrases such as "year three" or "year four".
ROLLING_ORIGIN_FOLDS = (
    (2018, 2019, "train_2017_18_to_2018_19__validate_2019_20"),
    (2019, 2020, "train_2017_18_to_2019_20__validate_2020_21"),
    (2020, 2021, "train_2017_18_to_2020_21__validate_2021_22"),
)
FINAL_TEST_YEAR = 2022

# Each activity is comparable for Question 3 only when all four variables exist:
#   MONTHS_12_      = participated during the last 12 months;
#   DAYS10P60GR_    = recent participation frequency/category;
#   INOUTA_         = indoor location indicator;
#   INOUTB_         = outdoor location indicator.
PREFIXES = ("MONTHS_12_", "DAYS10P60GR_", "INOUTA_", "INOUTB_")

# These four states are mutually exclusive and exhaustive.  Modelling the four
# states together is safer than fitting separate indoor and outdoor models,
# because one person can report both indoor and outdoor participation.
STATE_NAMES = ("neither_recorded", "indoor_only", "outdoor_only", "both")
STATE_COLS = tuple(f"{x}_rate" for x in STATE_NAMES)
LAG_COLS = tuple(f"lag_{x}" for x in STATE_COLS)

# Map each source file to the start year of its Active Lives survey wave.
# For example, survey_year=2017 means the 2017/18 wave, not calendar year 2017.
FILE_WAVES = {
    "2017_data_179_activities.csv": 2017,
    "2018_data_179_activities.csv": 2018,
    "1920_london32_stable179.csv": 2019,
    "2021_london32_stable179.csv": 2020,
    "year7_179activities.csv": 2021,
    "year8_179activities.csv": 2022,
}

ACTIVITY_LABELS = {0: "Inactive", 1: "Insufficiently Active", 2: "Active"}
INNER_OUTER_LABELS = {1: "Inner London", 2: "Outer London"}

# These ordered contrasts directly answer the second sentence of Question 3.
# A positive difference means that group A is more indoor-oriented than group B.
ACTIVITY_CONTRASTS = (
    (2, 0, "Active minus Inactive"),
    (2, 1, "Active minus Insufficiently Active"),
    (1, 0, "Insufficiently Active minus Inactive"),
)

# Convert the numeric LA_2023 value stored in the survey into both the official
# borough name and its ONS-style borough code.  This also allows the script to
# stop immediately if an unexpected/unmapped borough code appears.
BOROUGH_LOOKUP = {
    8: ("Barking and Dagenham", "E09000002"),
    9: ("Barnet", "E09000003"),
    17: ("Bexley", "E09000004"),
    30: ("Brent", "E09000005"),
    35: ("Bromley", "E09000006"),
    44: ("Camden", "E09000007"),
    68: ("Croydon", "E09000008"),
    78: ("Ealing", "E09000009"),
    91: ("Enfield", "E09000010"),
    107: ("Greenwich", "E09000011"),
    109: ("Hackney", "E09000012"),
    112: ("Hammersmith and Fulham", "E09000013"),
    114: ("Haringey", "E09000014"),
    117: ("Harrow", "E09000015"),
    122: ("Havering", "E09000016"),
    126: ("Hillingdon", "E09000017"),
    129: ("Hounslow", "E09000018"),
    135: ("Islington", "E09000019"),
    136: ("Kensington and Chelsea", "E09000020"),
    139: ("Kingston upon Thames", "E09000021"),
    142: ("Lambeth", "E09000022"),
    147: ("Lewisham", "E09000023"),
    160: ("Merton", "E09000024"),
    171: ("Newham", "E09000025"),
    196: ("Redbridge", "E09000026"),
    201: ("Richmond upon Thames", "E09000027"),
    241: ("Southwark", "E09000028"),
    255: ("Sutton", "E09000029"),
    272: ("Tower Hamlets", "E09000030"),
    279: ("Waltham Forest", "E09000031"),
    280: ("Wandsworth", "E09000032"),
    293: ("Westminster", "E09000033"),
}


In [3]:
# =============================================================================
# STEP 2 — Read command-line paths
# =============================================================================
def parse_args() -> argparse.Namespace:
    """Read the input and output paths.

    The six CSV files, 179.xlsx, the notebook and this Python file are stored
    in the same folder. Therefore, the default input directory is Path(".").

    Jupyter automatically passes its own kernel argument, such as
    --f=kernel-xxxx.json. parse_known_args() ignores this Jupyter-only argument
    and prevents the program from stopping with SystemExit: 2.

    Example command-line usage:
        python q3_indoor_outdoor_forecasting.py \
            --input-dir . \
            --output-dir q3_outputs \
            --bootstrap-reps 500
    """
    parser = argparse.ArgumentParser(
        description=(
            "Analyse and forecast indoor and outdoor activity participation "
            "for London and its 32 boroughs."
        )
    )

    # The input files are in the same folder as the notebook and Python file.
    parser.add_argument(
        "--input-dir",
        type=Path,
        default=Path("."),
        help=(
            "Folder containing the six CSV files and 179.xlsx. "
            "The default is the current working directory."
        ),
    )

    # The program will create this folder automatically if it does not exist.
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("q3_outputs"),
        help="Folder in which CSV results, plots and audit files are saved.",
    )

    # Number of Bayesian-bootstrap repetitions used for the formal comparison
    # of indoor/outdoor preferences between activity-level groups.
    parser.add_argument(
        "--bootstrap-reps",
        type=int,
        default=500,
        help=(
            "Bayesian-bootstrap repetitions for observed activity-level "
            "contrasts."
        ),
    )

    # Jupyter passes an internal argument such as --f=kernel-xxxx.json.
    # parse_known_args() reads our arguments and safely ignores Jupyter's
    # internal arguments.
    args, unknown_args = parser.parse_known_args()

    # Convert relative paths into paths based on the current working directory.
    args.input_dir = args.input_dir.resolve()
    args.output_dir = args.output_dir.resolve()

    if args.bootstrap_reps < 1:
        parser.error("--bootstrap-reps must be at least 1.")

    return args


def valid_numeric(frame: pd.DataFrame, allowed: set[int]) -> pd.DataFrame:
    """Convert survey values to numbers and remove invalid survey codes.

    Survey files may contain SPSS missing-value or routing codes such as
    -99, -98 or 97. These values must not be treated as genuine responses.

    This function converts values to numeric form and retains only the valid
    codes supplied for the relevant variable family. All other values become
    missing values (NaN).
    """
    # Blank/non-numeric/illegal values become NaN and are not imputed.
    # A valid zero remains zero; all-zero columns are not automatically dropped.
    out = frame.apply(pd.to_numeric, errors="coerce")
    return out.where(out.isin(allowed))


def wave_month_to_date(month_code: pd.Series) -> pd.Series:
    """Translate the continuous survey month code into a calendar date.

    Active Lives uses:
        month code 1  = November 2015
        month code 25 = November 2017
        month code 96 = October 2023

    The returned date is the first day of each month. Using a consistent
    monthly date is necessary for chronological sorting, monthly lags,
    train-validation-test splitting and plotting.
    """
    code = pd.to_numeric(month_code, errors="coerce").astype("Int64")

    year = 2015 + ((code + 9) // 12)
    month = ((code + 9) % 12) + 1

    return pd.to_datetime(
        {
            "year": year.astype("float"),
            "month": month.astype("float"),
            "day": 1,
        },
        errors="coerce",
    )


In [4]:
# =============================================================================
# STEP 3 — Find activities that are genuinely comparable across all six waves
# =============================================================================
def find_comparable_activities(input_dir: Path) -> tuple[list[str], pd.DataFrame]:
    """Return stable activity suffixes having all four required DVs in every CSV.

    The workbook may identify 179 stable activities, but an activity can be used
    for indoor/outdoor analysis only if Months, Days, Indoor and Outdoor columns
    all exist in all six survey waves.  The audit file records this decision.
    """
    # The 'Stable composites' sheet supplies the intended stable activity list.
    dictionary = pd.read_excel(input_dir / "179.xlsx", sheet_name="Stable composites")
    stable = dictionary["DV suffix"].dropna().astype(str).str.strip().tolist()

    # Reading only headers is fast and avoids loading six large data files merely
    # to check whether a column exists.
    headers = {
        name: set(pd.read_csv(input_dir / name, nrows=0).columns)
        for name in FILE_WAVES
    }
    # Keep the strict intersection: four fields x six waves must all be present.
    comparable = [
        suffix
        for suffix in stable
        if all(
            all(prefix + suffix in headers[name] for prefix in PREFIXES)
            for name in FILE_WAVES
        )
    ]
    # Save a transparent yes/no flag for every dictionary activity.
    audit = dictionary.copy()
    audit["four_DVs_present_all_six"] = audit["DV suffix"].isin(comparable)
    return comparable, audit


In [5]:
# =============================================================================
# STEP 4 — Convert one survey wave into respondent-level Q3 indicators
# =============================================================================
def derive_one_wave(
    path: Path, survey_year: int, activities: list[str]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Clean one CSV and return respondent indicators plus a missingness audit."""
    # First check the required identification, time, geography, activity-level
    # and weighting variables before loading the full set of selected columns.
    header = set(pd.read_csv(path, nrows=0).columns)
    core = [
        "serial", "year", "month", "LA_2023", "LondInOut", "Reg9",
        "MEMS7GR_ALL", "wt_final", "wt_time",
    ]
    missing_core = [c for c in core if c not in header]
    if missing_core:
        raise ValueError(f"{path.name}: missing core columns {missing_core}")
    activity_cols = [prefix + suffix for suffix in activities for prefix in PREFIXES]
    usecols = core + activity_cols
    df = pd.read_csv(path, usecols=usecols, low_memory=False)

    # Apply each variable family's own legal coding.  Invalid/routing values
    # become NaN and therefore cannot accidentally count as participation.
    family_specs = {
        "MONTHS_12": (["MONTHS_12_" + s for s in activities], {0, 1}),
        "DAYS10P60GR": (["DAYS10P60GR_" + s for s in activities], {0, 1, 2}),
        "INOUTA_indoor": (["INOUTA_" + s for s in activities], {0, 1}),
        "INOUTB_outdoor": (["INOUTB_" + s for s in activities], {0, 1}),
    }
    cleaned: dict[str, np.ndarray] = {}
    audit_rows: list[dict[str, Any]] = []
    for family, (columns, allowed) in family_specs.items():
        raw = df[columns]
        numeric = raw.apply(pd.to_numeric, errors="coerce")
        valid = numeric.where(numeric.isin(allowed))
        audit_rows.append({
            "source_file": path.name,
            "survey_wave": f"{survey_year}/{str(survey_year + 1)[-2:]}",
            "variable_family": family,
            "cells_total": int(raw.size),
            "raw_missing_or_blank": int(raw.isna().to_numpy().sum()),
            "invalid_nonmissing_to_nan": int((raw.notna() & valid.isna()).to_numpy().sum()),
            "valid_zero_retained": int((valid == 0).to_numpy().sum()),
            "valid_nonzero_retained": int((valid > 0).to_numpy().sum()),
            "processing_rule": "Blank/invalid -> pandas NaN; valid zero remains 0; no imputation",
        })
        cleaned[family] = valid.to_numpy()
    months = cleaned["MONTHS_12"]
    days = cleaned["DAYS10P60GR"]
    indoor = cleaned["INOUTA_indoor"]
    outdoor = cleaned["INOUTB_outdoor"]

    # Core Q3 participation rule for person i and activity j:
    #   participated_ij = 1 when MONTHS_12_ij == 1 AND DAYS10P60GR_ij > 0.
    # Indoor/outdoor status is counted only for activities satisfying this rule.
    # Comparisons with NaN are False, so missing values never create a
    # positive participation flag; no mean/mode imputation is applied.
    participated = (months == 1) & np.isin(days, [1, 2])
    indoor_flags = participated & (indoor == 1)
    outdoor_flags = participated & (outdoor == 1)

    # Reduce the activity-by-person matrices to one set of person-level flags.
    # any() avoids incorrectly treating several activities by one respondent as
    # several people.
    any_participated = participated.any(axis=1)
    any_indoor = indoor_flags.any(axis=1)
    any_outdoor = outdoor_flags.any(axis=1)

    # Retain the core survey information and add readable time/geography labels.
    out = df[core].copy()
    out["survey_year"] = survey_year
    out["survey_wave"] = f"{survey_year}/{str(survey_year + 1)[-2:]}"
    out["period_start"] = wave_month_to_date(out["month"])
    out["borough_id"] = pd.to_numeric(out["LA_2023"], errors="coerce").astype("Int64")
    out["borough_name"] = out["borough_id"].map(
        {key: value[0] for key, value in BOROUGH_LOOKUP.items()}
    )
    out["borough_code"] = out["borough_id"].map(
        {key: value[1] for key, value in BOROUGH_LOOKUP.items()}
    )
    # An unmapped value would make borough-level results ambiguous, so fail fast.
    unknown_boroughs = sorted(
        out.loc[out["borough_id"].notna() & out["borough_name"].isna(), "borough_id"]
        .astype(int).unique().tolist()
    )
    if unknown_boroughs:
        raise ValueError(f"{path.name}: unmapped LA_2023 codes {unknown_boroughs}")
    out["inner_outer_code"] = pd.to_numeric(out["LondInOut"], errors="coerce").astype("Int64")
    out["inner_outer"] = out["inner_outer_code"].map(INNER_OUTER_LABELS)
    # MEMS7GR_ALL defines the respondent's overall activity-level group.  It is
    # used for stratification, not reconstructed from the 146/179 activity items.
    out["activity_level_code"] = pd.to_numeric(out["MEMS7GR_ALL"], errors="coerce")
    out.loc[~out["activity_level_code"].isin(ACTIVITY_LABELS), "activity_level_code"] = np.nan
    out["activity_level"] = out["activity_level_code"].map(ACTIVITY_LABELS)
    out["wt_final"] = pd.to_numeric(out["wt_final"], errors="coerce")
    out["wt_time"] = pd.to_numeric(out["wt_time"], errors="coerce")
    # serial may repeat in different waves, so prefix it with survey_year.
    out["respondent_id"] = out["survey_year"].astype(str) + "_" + out["serial"].astype(str)
    out["any_recent_comparable_activity"] = any_participated.astype("int8")
    out["any_indoor"] = any_indoor.astype("int8")
    out["any_outdoor"] = any_outdoor.astype("int8")
    # Create exactly one of four location states for each respondent.
    out["both"] = (any_indoor & any_outdoor).astype("int8")
    out["indoor_only"] = (any_indoor & ~any_outdoor).astype("int8")
    out["outdoor_only"] = (~any_indoor & any_outdoor).astype("int8")
    out["neither_recorded"] = (~any_indoor & ~any_outdoor).astype("int8")
    # Split "neither recorded" into its two substantively different causes.
    # It must never be described as physical inactivity: MEMS7GR_ALL supplies
    # the separate Inactive/Fairly Active/Active grouping used in the study.
    out["no_qualifying_comparable_activity"] = (~any_participated).astype("int8")
    out["recent_activity_without_recorded_location"] = (
        any_participated & ~any_indoor & ~any_outdoor
    ).astype("int8")
    out["n_recent_activity_flags"] = participated.sum(axis=1).astype("int16")
    out["n_indoor_flags"] = indoor_flags.sum(axis=1).astype("int16")
    out["n_outdoor_flags"] = outdoor_flags.sum(axis=1).astype("int16")
    if not np.array_equal(
        out["neither_recorded"].to_numpy(),
        (out["no_qualifying_comparable_activity"] + out["recent_activity_without_recorded_location"]).to_numpy(),
    ):
        raise AssertionError("Neither-recorded subcategories do not reconcile")
    return out, pd.DataFrame(audit_rows)


In [6]:
# =============================================================================
# STEP 5 — Combine all six waves and validate the respondent dataset
# =============================================================================
def make_respondent_data(input_dir: Path, output_dir: Path) -> tuple[pd.DataFrame, list[str]]:
    """Build and save the complete 2017/18--2022/23 respondent-level file."""
    activities, activity_audit = find_comparable_activities(input_dir)
    activity_audit.to_csv(output_dir / "activity_comparability_audit.csv", index=False)
    waves = []
    preprocessing_audits = []
    for filename, survey_year in FILE_WAVES.items():
        print(f"Reading {filename} ...", flush=True)
        wave, preprocessing_audit = derive_one_wave(
            input_dir / filename, survey_year, activities
        )
        waves.append(wave)
        preprocessing_audits.append(preprocessing_audit)
    respondents = pd.concat(waves, ignore_index=True)
    pd.concat(preprocessing_audits, ignore_index=True).to_csv(
        output_dir / "preprocessing_missing_zero_audit.csv", index=False
    )
    category_counts = respondents.groupby(
        ["survey_year", "survey_wave"], as_index=False
    )[
        [
            "neither_recorded",
            "no_qualifying_comparable_activity",
            "recent_activity_without_recorded_location",
            "indoor_only",
            "outdoor_only",
            "both",
        ]
    ].sum()
    category_counts["neither_reconciliation_error"] = (
        category_counts["neither_recorded"]
        - category_counts["no_qualifying_comparable_activity"]
        - category_counts["recent_activity_without_recorded_location"]
    )
    category_counts.to_csv(
        output_dir / "neither_recorded_definition_counts.csv", index=False
    )

    # Every survey wave must contain all 12 expected months.  This check catches
    # an incorrect month-code conversion before any monthly modelling begins.
    expected_dates = {
        year: pd.date_range(f"{year}-11-01", f"{year + 1}-10-01", freq="MS")
        for year in FILE_WAVES.values()
    }
    for year, dates in expected_dates.items():
        got = set(respondents.loc[respondents.survey_year == year, "period_start"].dropna())
        if got != set(dates):
            raise AssertionError(f"Wave {year} month mapping is incomplete or incorrect")

    # A valid respondent must belong to one and only one of the four states.
    state_sum = respondents[list(STATE_NAMES)].sum(axis=1)
    if not (state_sum == 1).all():
        raise AssertionError("Four respondent states are not mutually exclusive/exhaustive")
    respondents.to_csv(output_dir / "q3_respondent_2017_2022.csv", index=False)
    return respondents, activities


In [7]:
# =============================================================================
# STEP 6 — Aggregate respondents into weighted borough panels
# =============================================================================
def aggregate_panel(df: pd.DataFrame, group_cols: list[str], weight_col: str) -> pd.DataFrame:
    """Calculate four weighted state proportions for each requested panel cell.

    Annual panels use wt_final; monthly panels use wt_time.  Survey weights are
    applied here exactly once.  Later models use effective_n only as a measure
    of how reliable each already-weighted panel estimate is.
    """
    needed = group_cols + list(STATE_NAMES) + [weight_col]
    work = df[needed].copy()
    work[weight_col] = pd.to_numeric(work[weight_col], errors="coerce")
    # A panel estimate requires complete grouping keys and a positive finite
    # survey weight.  Invalid weights are excluded rather than replaced by 1.
    work = work[
        work[group_cols].notna().all(axis=1)
        & work[weight_col].notna()
        & np.isfinite(work[weight_col])
        & (work[weight_col] > 0)
    ].copy()
    # Store squared weights for Kish effective sample size, and weighted state
    # indicators for the numerators of each weighted proportion.
    work["_w2"] = work[weight_col] ** 2
    for state in STATE_NAMES:
        work[f"_wy_{state}"] = work[weight_col] * work[state]
    g = work.groupby(group_cols, observed=True, dropna=False)
    panel = g.agg(
        raw_n=(weight_col, "size"),
        weight_sum=(weight_col, "sum"),
        weight_sq_sum=("_w2", "sum"),
    ).reset_index()
    for state in STATE_NAMES:
        num = g[f"_wy_{state}"].sum().reset_index(name="_num")
        panel = panel.merge(num, on=group_cols, how="left")
        panel[f"{state}_rate"] = panel.pop("_num") / panel["weight_sum"]
    # Kish effective n = (sum w)^2 / sum(w^2).  It reflects how much information
    # remains after unequal survey weighting and can be lower than raw_n.
    panel["effective_n"] = panel["weight_sum"] ** 2 / panel["weight_sq_sum"]
    panel["low_effective_n_10"] = panel["effective_n"] < 10
    panel["low_effective_n_20"] = panel["effective_n"] < 20
    # Derived rates answer the intuitive indoor/outdoor questions.  They overlap
    # because respondents in 'both' contribute to each rate.
    panel["indoor_rate"] = panel["indoor_only_rate"] + panel["both_rate"]
    panel["outdoor_rate"] = panel["outdoor_only_rate"] + panel["both_rate"]
    # Exposure shares provide a two-part indoor-vs-outdoor balance that sums to
    # one.  A 'both' respondent contributes one indoor and one outdoor exposure.
    denominator = (
        panel["indoor_only_rate"] + panel["outdoor_only_rate"] + 2 * panel["both_rate"]
    )
    panel["indoor_exposure_share"] = (
        (panel["indoor_only_rate"] + panel["both_rate"]) / denominator.replace(0, np.nan)
    )
    panel["outdoor_exposure_share"] = 1 - panel["indoor_exposure_share"]
    if not np.allclose(panel[list(STATE_COLS)].sum(axis=1), 1, atol=1e-9):
        raise AssertionError("Panel composition does not sum to one")
    return panel


In [8]:
# =============================================================================
# STEP 7 — Stabilise noisy borough cells with transparent partial pooling
# =============================================================================
def shrink_composition(
    panel: pd.DataFrame,
    time_cols: list[str],
    prior_strength: float,
) -> pd.DataFrame:
    """Partial pooling toward the same-time London activity-level composition.

    Raw direct estimates remain in *_rate_raw. The regularised *_rate columns
    are used by models. effective_n controls the amount of pooling.

    Small borough x activity-level cells can have direct estimates of exactly
    0 or 1 by chance.  Each estimate is therefore pulled toward the London-wide
    composition for the same time and activity level.  Larger effective_n means
    less shrinkage and hence more trust in the borough's direct estimate.
    """
    out = panel.copy()
    for col in STATE_COLS:
        out[f"{col}_raw"] = out[col]
        out[f"_weighted_{col}"] = out["weight_sum"] * out[col]
    # The prior is London-wide but time- and activity-level-specific, so genuine
    # temporal change and differences among activity groups are still retained.
    prior_keys = time_cols + ["activity_level_code"]
    g = out.groupby(prior_keys, observed=True)
    priors = g["weight_sum"].sum().reset_index(name="_prior_weight")
    for col in STATE_COLS:
        num = g[f"_weighted_{col}"].sum().reset_index(name=f"_prior_num_{col}")
        priors = priors.merge(num, on=prior_keys, how="left")
        priors[f"_prior_{col}"] = priors[f"_prior_num_{col}"] / priors["_prior_weight"]
    out = out.merge(
        priors[prior_keys + [f"_prior_{c}" for c in STATE_COLS]],
        on=prior_keys,
        how="left",
        validate="many_to_one",
    )
    out["shrinkage_prior_strength"] = float(prior_strength)
    # Reliability r is near 1 for a well-supported cell and near 0 for a very
    # small cell.  The smoothed estimate is r*borough + (1-r)*London prior.
    out["direct_estimate_reliability"] = out["effective_n"] / (out["effective_n"] + prior_strength)
    r = out["direct_estimate_reliability"]
    for col in STATE_COLS:
        out[col] = r * out[f"{col}_raw"] + (1 - r) * out[f"_prior_{col}"]
    drop_cols = [c for c in out if c.startswith("_weighted_") or c.startswith("_prior_")]
    out = out.drop(columns=drop_cols)
    out["indoor_rate"] = out["indoor_only_rate"] + out["both_rate"]
    out["outdoor_rate"] = out["outdoor_only_rate"] + out["both_rate"]
    denominator = out["indoor_only_rate"] + out["outdoor_only_rate"] + 2 * out["both_rate"]
    out["indoor_exposure_share"] = (out["indoor_only_rate"] + out["both_rate"]) / denominator.replace(0, np.nan)
    out["outdoor_exposure_share"] = 1 - out["indoor_exposure_share"]
    if not np.allclose(out[list(STATE_COLS)].sum(axis=1), 1, atol=1e-9):
        raise AssertionError("Shrunk composition does not sum to one")
    return out


def make_panels(respondents: pd.DataFrame, output_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Create annual and monthly borough-by-activity-level modelling panels."""
    common = [
        "borough_id", "borough_name", "borough_code",
        "inner_outer_code", "inner_outer", "activity_level_code", "activity_level",
    ]
    # Annual estimates use the final annual weight; monthly estimates use the
    # dedicated time-series weight supplied by the survey.
    annual = aggregate_panel(respondents, ["survey_year", "survey_wave"] + common, "wt_final")
    monthly = aggregate_panel(respondents, ["period_start", "survey_year", "survey_wave"] + common, "wt_time")
    # Monthly cells are much smaller, so they use stronger pooling (20 vs 5).
    # The original direct rates remain available in *_rate_raw columns.
    annual = shrink_composition(
        annual, ["survey_year"], prior_strength=ANNUAL_POOLING_STRENGTH
    )
    monthly = shrink_composition(
        monthly, ["period_start"], prior_strength=MONTHLY_POOLING_STRENGTH
    )
    annual.to_csv(output_dir / "q3_borough_activitylevel_annual.csv", index=False)
    monthly.to_csv(output_dir / "q3_borough_activitylevel_monthly.csv", index=False)
    return annual, monthly


In [9]:
# =============================================================================
# STEP 8 — Transform the four-part outcome before regression
# =============================================================================
def alr_transform(probabilities: np.ndarray, epsilon: float = ALR_EPS) -> np.ndarray:
    """Convert four proportions summing to one into three unconstrained values.

    Ordinary regressors can otherwise predict negative rates or rates whose sum
    is not one.  The additive log-ratio (ALR) uses 'both' as the reference part.
    """
    p = np.clip(np.asarray(probabilities, dtype=float), epsilon, None)
    p = p / p.sum(axis=1, keepdims=True)
    return np.log(p[:, :3] / p[:, [3]])


def alr_inverse(z: np.ndarray) -> np.ndarray:
    """Convert three ALR predictions back to four valid probabilities."""
    z = np.asarray(z, dtype=float)
    logits = np.column_stack([z, np.zeros(len(z))])
    logits -= logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


In [10]:
# =============================================================================
# STEP 9 — Build time, category, lag and COVID-control predictors
# =============================================================================
def add_model_features(panel: pd.DataFrame, frequency: str) -> pd.DataFrame:
    """Add predictors without using any future outcome information.

    COVID variables are controls for an unusual shock inside the six-year data;
    they do not restrict the sample to the COVID period and do not redefine Q3.
    """
    out = panel.copy()
    # Canonical integer-like strings prevent observed "0.0" versus future "0"
    # mismatches that OneHotEncoder would otherwise treat as unseen categories.
    out["borough_cat"] = pd.to_numeric(out["borough_id"], errors="coerce").astype("Int64").astype(str)
    out["activity_cat"] = pd.to_numeric(out["activity_level_code"], errors="coerce").astype("Int64").astype(str)
    out["inner_outer_cat"] = pd.to_numeric(out["inner_outer_code"], errors="coerce").astype("Int64").astype(str)
    # Lags are calculated separately for every borough and activity-level group.
    group = ["borough_id", "activity_level_code"]
    out = out.sort_values(group + (["survey_year"] if frequency == "annual" else ["period_start"]))
    if frequency == "annual":
        # Annual naive information is the previous survey wave (lag 1 year).
        for state_col, lag_col in zip(STATE_COLS, LAG_COLS):
            out[lag_col] = out.groupby(group, observed=True)[state_col].shift(1)
    else:
        # Calendar join, not shift(12): some low-n borough/activity cells have
        # no row in a month, so row offsets would silently select the wrong date.
        # Monthly naive information is the same calendar month one year earlier.
        lag_table = out[group + ["period_start"] + list(STATE_COLS)].copy()
        lag_table["period_start"] = pd.to_datetime(lag_table["period_start"]) + pd.DateOffset(years=1)
        lag_table = lag_table.rename(columns=dict(zip(STATE_COLS, LAG_COLS)))
        out = out.merge(lag_table, on=group + ["period_start"], how="left", validate="one_to_one")

    if frequency == "annual":
        # covid_era captures a possible post-2020 level shift;
        # covid_disruption isolates the unusually affected 2020/21 wave;
        # time_since_covid permits a different slope after the shock.
        out["trend"] = out["survey_year"] - 2017
        out["covid_era"] = (out["survey_year"] >= 2020).astype(float)
        out["covid_disruption"] = (out["survey_year"] == 2020).astype(float)
        out["time_since_covid"] = np.maximum(out["survey_year"] - 2020, 0)
    else:
        # Monthly models also need sine/cosine terms for repeating seasonality.
        date = pd.to_datetime(out["period_start"])
        out["trend"] = (date.dt.year - 2017) * 12 + date.dt.month - 11
        out["month_sin"] = np.sin(2 * np.pi * (date.dt.month - 1) / 12)
        out["month_cos"] = np.cos(2 * np.pi * (date.dt.month - 1) / 12)
        # The monthly disruption window is explicitly dated March 2020--July
        # 2021; all pre- and post-window observations remain in the analysis.
        covid_start = pd.Timestamp("2020-03-01")
        disruption_end = pd.Timestamp("2021-07-01")
        out["covid_era"] = (date >= covid_start).astype(float)
        out["covid_disruption"] = ((date >= covid_start) & (date <= disruption_end)).astype(float)
        delta = (date.dt.year - 2020) * 12 + date.dt.month - 3
        out["time_since_covid"] = np.maximum(delta, 0)
    return out


def feature_columns(frequency: str) -> tuple[list[str], list[str]]:
    """Return categorical and numeric predictors expected by every ML model."""
    categorical = ["borough_cat", "activity_cat", "inner_outer_cat"]
    numeric = ["trend", "covid_era", "covid_disruption", "time_since_covid"] + list(LAG_COLS)
    if frequency == "monthly":
        numeric += ["month_sin", "month_cos"]
    return categorical, numeric


In [11]:
# =============================================================================
# STEP 10 — Define the three trainable pipelines (Naive is added in Step 12)
# =============================================================================
def build_pipeline(model_name: str, params: dict[str, Any], frequency: str) -> Pipeline:
    """Create preprocessing and one requested regression model as one pipeline."""
    categorical, numeric = feature_columns(frequency)
    # Borough/activity categories are one-hot encoded. Numeric predictors are
    # standardised, which is essential for Ridge and harmless for tree models.
    preprocessor = ColumnTransformer(
        [
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
            ("num", StandardScaler(), numeric),
        ],
        remainder="drop",
    )
    # All models predict the three ALR coordinates; Gradient Boosting needs an
    # explicit multi-output wrapper because its base estimator is single-output.
    if model_name == "Ridge":
        model = Ridge(alpha=params["alpha"])
    elif model_name == "Random Forest":
        model = RandomForestRegressor(
            n_estimators=params.get("n_estimators", 250),
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    elif model_name == "Gradient Boosting":
        base = GradientBoostingRegressor(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            min_samples_leaf=params.get("min_samples_leaf", 5),
            random_state=RANDOM_STATE,
            loss="huber",
        )
        model = MultiOutputRegressor(base, n_jobs=-1)
    else:
        raise ValueError(model_name)
    return Pipeline([("preprocess", preprocessor), ("model", model)])


def model_grid(frequency: str) -> dict[str, list[dict[str, Any]]]:
    """List the candidate hyperparameters compared on the validation wave."""
    # Compact grids keep the supplied full-data run reproducible on a laptop.
    return {
        "Ridge": [{"alpha": x} for x in (0.1, 1.0, 10.0, 100.0)],
        "Random Forest": [
            {"n_estimators": 250, "max_depth": 8, "min_samples_leaf": 5},
            {"n_estimators": 250, "max_depth": None, "min_samples_leaf": 10},
        ],
        "Gradient Boosting": [
            {"n_estimators": 120, "learning_rate": 0.03, "max_depth": 2, "min_samples_leaf": 5},
            {"n_estimators": 180, "learning_rate": 0.03, "max_depth": 2, "min_samples_leaf": 10},
            {"n_estimators": 120, "learning_rate": 0.05, "max_depth": 3, "min_samples_leaf": 10},
        ],
    }


In [12]:
# =============================================================================
# STEP 11 — Evaluate forecasts using the agreed composition metrics
# =============================================================================
def metric_row(y: np.ndarray, pred: np.ndarray, weights: np.ndarray) -> dict[str, float]:
    """Calculate MAE, Total Variation and supporting error measures.

    Total Variation (TV) is half the sum of absolute errors across the four
    states, so it measures the amount of probability mass placed in the wrong
    categories. MAE reports the average component-wise error on the original
    proportion scale and is directly interpretable in percentage points.
    Together they assess global compositional fidelity and typical error size.
    """
    weights = np.asarray(weights, dtype=float)
    tv = 0.5 * np.abs(y - pred).sum(axis=1)
    actual_in = y[:, 1] + y[:, 3]
    pred_in = pred[:, 1] + pred[:, 3]
    actual_out = y[:, 2] + y[:, 3]
    pred_out = pred[:, 2] + pred[:, 3]
    indoor_outdoor_abs_error = np.column_stack([
        np.abs(actual_in - pred_in),
        np.abs(actual_out - pred_out),
    ])
    result: dict[str, float] = {
        "weighted_tv": float(np.average(tv, weights=weights)),
        "unweighted_tv": float(tv.mean()),
        "weighted_mae": float(
            np.average(indoor_outdoor_abs_error.mean(axis=1), weights=weights)
        ),
        "unweighted_mae": float(indoor_outdoor_abs_error.mean()),
        "weighted_component_mae": float(
            np.average(np.abs(y - pred).mean(axis=1), weights=weights)
        ),
    }
    # Report state-specific MAE/RMSE/R2 as diagnostics, while TV remains the
    # main model-selection criterion.
    for i, state in enumerate(STATE_NAMES):
        result[f"{state}_weighted_mae"] = float(np.average(np.abs(y[:, i] - pred[:, i]), weights=weights))
        result[f"{state}_weighted_rmse"] = float(np.sqrt(np.average((y[:, i] - pred[:, i]) ** 2, weights=weights)))
        result[f"{state}_weighted_r2"] = float(r2_score(y[:, i], pred[:, i], sample_weight=weights))
    # Indoor and outdoor rates are derived by adding the shared 'both' state.
    result["indoor_weighted_mae"] = float(np.average(np.abs(actual_in - pred_in), weights=weights))
    result["outdoor_weighted_mae"] = float(np.average(np.abs(actual_out - pred_out), weights=weights))
    result["indoor_weighted_rmse"] = float(np.sqrt(np.average((actual_in - pred_in) ** 2, weights=weights)))
    result["outdoor_weighted_rmse"] = float(np.sqrt(np.average((actual_out - pred_out) ** 2, weights=weights)))
    return result


def fit_pipeline(
    model_name: str,
    params: dict[str, Any],
    frequency: str,
    train: pd.DataFrame,
    epsilon: float = ALR_EPS,
) -> Pipeline:
    """Fit one non-naive model using effective sample size as sample weight."""
    categorical, numeric = feature_columns(frequency)
    model = build_pipeline(model_name, params, frequency)
    x = train[categorical + numeric]
    # Transform the four proportions before fitting so predictions remain a
    # valid four-part composition after alr_inverse().
    y = alr_transform(train[list(STATE_COLS)].to_numpy(), epsilon=epsilon)
    w = train["effective_n"].to_numpy()
    if model_name == "Gradient Boosting":
        model.fit(x, y, model__sample_weight=w)
    else:
        model.fit(x, y, model__sample_weight=w)
    return model


def predict_pipeline(model: Pipeline, data: pd.DataFrame, frequency: str) -> np.ndarray:
    """Predict and immediately convert ALR outputs to four probabilities."""
    categorical, numeric = feature_columns(frequency)
    return alr_inverse(model.predict(data[categorical + numeric]))


@dataclass
class ModelSelection:
    """Keep the selected specification, fitted final model and all metrics."""
    frequency: str
    chosen_name: str
    chosen_params: dict[str, Any]
    final_model: Pipeline | None
    metrics: pd.DataFrame
    epsilon: float = ALR_EPS


In [13]:
# =============================================================================
# STEP 12 — Train, validate, select, test, and refit the four model families
# =============================================================================
def select_and_test_models(
    features: pd.DataFrame,
    frequency: str,
    epsilon: float = ALR_EPS,
) -> ModelSelection:
    """Select by three rolling-origin folds, then evaluate 2022/23 once.

    Fold 1: train 2017/18--2018/19, validate 2019/20.
    Fold 2: train 2017/18--2019/20, validate 2020/21.
    Fold 3: train 2017/18--2020/21, validate 2021/22.
    Final:  refit through 2021/22 and test once on 2022/23.

    The selected candidate minimises the mean rank across mean weighted TV and
    mean weighted MAE.  TV evaluates the full four-part distribution, while
    MAE gives the interpretable average error of the derived indoor and outdoor
    participation rates. The final test is never
    inspected during model or hyperparameter selection.
    """
    # Lagged models cannot use the first observation of a group because no
    # previous year/same month exists.  Other missing target/reliability rows are
    # also removed here, after the panel itself has been saved for auditing.
    clean = features.dropna(subset=list(LAG_COLS) + list(STATE_COLS) + ["effective_n"]).copy()
    test = clean[clean["survey_year"] == FINAL_TEST_YEAR]
    if test.empty:
        raise ValueError(f"{frequency}: empty 2022/23 final test split")

    candidates: list[tuple[str, dict[str, Any]]] = [("Naive", {})]
    candidates.extend(
        (name, params)
        for name, configs in model_grid(frequency).items()
        for params in configs
    )
    fold_records: list[dict[str, Any]] = []
    for model_name, params in candidates:
        for train_end, valid_year, fold_label in ROLLING_ORIGIN_FOLDS:
            train = clean[clean["survey_year"].between(2017, train_end)]
            valid = clean[clean["survey_year"] == valid_year]
            if train.empty or valid.empty:
                raise ValueError(f"{frequency}: empty rolling fold {fold_label}")
            y_valid = valid[list(STATE_COLS)].to_numpy()
            if model_name == "Naive":
                pred = valid[list(LAG_COLS)].to_numpy()
            else:
                fitted = fit_pipeline(
                    model_name, params, frequency, train, epsilon=epsilon
                )
                pred = predict_pipeline(fitted, valid, frequency)
            fold_records.append({
                "frequency": frequency,
                "split": "rolling_validation_fold",
                "fold": fold_label,
                "train_start_wave": "2017/18",
                "train_end_wave": f"{train_end}/{str(train_end + 1)[-2:]}",
                "validation_wave": f"{valid_year}/{str(valid_year + 1)[-2:]}",
                "model": model_name,
                "params": json.dumps(params, sort_keys=True),
                "alr_epsilon": epsilon,
                **metric_row(y_valid, pred, valid["effective_n"].to_numpy()),
            })

    fold_rows = pd.DataFrame(fold_records)
    group_cols = ["frequency", "model", "params", "alr_epsilon"]
    metric_cols = [c for c in fold_rows.columns if c.startswith("weighted_") or c.startswith("unweighted_")]
    cv_summary = fold_rows.groupby(group_cols, as_index=False)[metric_cols].mean()
    cv_summary["split"] = "rolling_validation_mean"
    cv_summary["fold"] = "mean_of_3_folds"
    cv_summary["train_start_wave"] = "2017/18"
    cv_summary["train_end_wave"] = "varies_by_fold"
    cv_summary["validation_wave"] = "2019/20, 2020/21, 2021/22"
    cv_summary["tv_rank"] = cv_summary["weighted_tv"].rank(method="min")
    cv_summary["mae_rank"] = cv_summary["weighted_mae"].rank(method="min")
    cv_summary["selection_mean_rank"] = (
        cv_summary["tv_rank"] + cv_summary["mae_rank"]
    ) / 2
    naive_cv_tv = float(
        cv_summary.loc[cv_summary.model == "Naive", "weighted_tv"].iloc[0]
    )
    cv_summary["skill_vs_naive"] = 1 - cv_summary["weighted_tv"] / naive_cv_tv
    winner = cv_summary.sort_values(
        ["selection_mean_rank", "weighted_tv", "weighted_mae", "model", "params"]
    ).iloc[0]
    chosen_name = str(winner["model"])
    chosen_params = json.loads(str(winner["params"]))
    cv_summary["is_selected_model"] = (
        (cv_summary["model"] == chosen_name)
        & (cv_summary["params"] == json.dumps(chosen_params, sort_keys=True))
    )

    # Hyperparameters are frozen after validation. Refit each model on train+valid,
    # then use 2022/23 once as the untouched final test.
    train_valid = clean[clean["survey_year"].between(2017, 2021)]
    y_test = test[list(STATE_COLS)].to_numpy()
    naive_test = test[list(LAG_COLS)].to_numpy()
    test_records = [{
        "frequency": frequency,
        "split": "final_test_2022_23",
        "fold": "untouched_final_test",
        "train_start_wave": "2017/18",
        "train_end_wave": "2021/22",
        "validation_wave": "2022/23",
        "model": "Naive",
        "params": "{}",
        "alr_epsilon": epsilon,
        **metric_row(y_test, naive_test, test["effective_n"].to_numpy()),
    }]
    if chosen_name != "Naive":
        fitted = fit_pipeline(
            chosen_name, chosen_params, frequency, train_valid, epsilon=epsilon
        )
        pred = predict_pipeline(fitted, test, frequency)
        test_records.append({
            "frequency": frequency,
            "split": "final_test_2022_23",
            "fold": "untouched_final_test",
            "train_start_wave": "2017/18",
            "train_end_wave": "2021/22",
            "validation_wave": "2022/23",
            "model": chosen_name,
            "params": json.dumps(chosen_params, sort_keys=True),
            "alr_epsilon": epsilon,
            **metric_row(y_test, pred, test["effective_n"].to_numpy()),
        })
    test_rows = pd.DataFrame(test_records)
    test_rows["alr_epsilon"] = epsilon
    naive_test_tv = test_rows.loc[test_rows.model == "Naive", "weighted_tv"].iloc[0]
    test_rows["skill_vs_naive"] = 1 - test_rows["weighted_tv"] / naive_test_tv
    metrics = pd.concat([fold_rows, cv_summary, test_rows], ignore_index=True, sort=False)

    # Finally refit the validation-selected winner on the complete six-year
    # history.  A Naive winner needs no fitted object.
    final_model = None
    if chosen_name != "Naive":
        final_model = fit_pipeline(
            chosen_name, chosen_params, frequency, clean, epsilon=epsilon
        )
    return ModelSelection(
        frequency, chosen_name, chosen_params, final_model, metrics, epsilon
    )


In [14]:
# =============================================================================
# STEP 13 — Specify future COVID-control sensitivity scenarios
# =============================================================================
def future_covid_features(row: dict[str, Any], frequency: str, scenario: str) -> dict[str, Any]:
    """Set future time/COVID controls without changing the main Q3 forecast.

    persistent_legacy keeps the post-COVID level/slope controls active.
    legacy_recovery sets only future COVID-specific controls to zero while
    preserving the ordinary long-term trend and monthly seasonality.

    The difference is a sensitivity range, not a statistical confidence band.
    """
    persistent = scenario == "persistent_legacy"
    if frequency == "annual":
        year = int(row["survey_year"])
        row["trend"] = year - 2017
        row["covid_era"] = float(persistent)
        row["covid_disruption"] = 0.0
        row["time_since_covid"] = float(max(year - 2020, 0) if persistent else 0)
    else:
        date = pd.Timestamp(row["period_start"])
        row["trend"] = (date.year - 2017) * 12 + date.month - 11
        row["month_sin"] = np.sin(2 * np.pi * (date.month - 1) / 12)
        row["month_cos"] = np.cos(2 * np.pi * (date.month - 1) / 12)
        row["covid_era"] = float(persistent)
        row["covid_disruption"] = 0.0
        delta = (date.year - 2020) * 12 + date.month - 3
        row["time_since_covid"] = float(max(delta, 0) if persistent else 0)
    return row


def recursive_forecast(
    observed: pd.DataFrame,
    selection: ModelSelection,
    frequency: str,
) -> pd.DataFrame:
    """Forecast 2023--2028 one step at a time for every borough/activity group.

    Each newly predicted composition is stored in history and becomes a lag for
    a later forecast.  This is necessary because true future lagged outcomes do
    not exist at prediction time.
    """
    # Keep one row of stable descriptive information for each of the 32 x 3
    # borough/activity-level groups that will be forecast.
    groups = (
        observed.sort_values("survey_year")
        .drop_duplicates(["borough_id", "activity_level_code"], keep="last")
        [[
            "borough_id", "borough_name", "borough_code",
            "inner_outer_code", "inner_outer", "activity_level_code", "activity_level",
        ]]
    )
    outputs = []
    # Run the identical selected model under two future COVID-control settings.
    scenarios = ("persistent_legacy", "legacy_recovery")
    for scenario in scenarios:
        # history maps (borough, activity level, period) to a four-part outcome.
        history: dict[tuple[int, int, Any], np.ndarray] = {}
        if frequency == "annual":
            last = observed[observed.survey_year == 2022]
            for _, r in last.iterrows():
                history[(int(r.borough_id), int(r.activity_level_code), 2022)] = r[list(STATE_COLS)].to_numpy(float)
            # Six future survey waves: 2023/24 through 2028/29.
            future_periods: list[Any] = list(range(2023, 2029))
        else:
            for _, r in observed.iterrows():
                history[(int(r.borough_id), int(r.activity_level_code), pd.Timestamp(r.period_start))] = r[list(STATE_COLS)].to_numpy(float)
            start = pd.Timestamp(observed.period_start.max()) + pd.offsets.MonthBegin(1)
            # Six years x 12 months = 72 monthly forecasts.
            future_periods = list(pd.date_range(start, periods=72, freq="MS"))

        for period in future_periods:
            rows = []
            for _, g in groups.iterrows():
                borough = int(g.borough_id)
                level = int(g.activity_level_code)
                if frequency == "annual":
                    lag_key = (borough, level, int(period) - 1)
                    survey_year = int(period)
                    period_start = pd.NaT
                    survey_wave = f"{survey_year}/{str(survey_year + 1)[-2:]}"
                else:
                    lag_date = pd.Timestamp(period) - pd.DateOffset(years=1)
                    lag_key = (borough, level, lag_date)
                    period_start = pd.Timestamp(period)
                    survey_year = period_start.year if period_start.month >= 11 else period_start.year - 1
                    survey_wave = f"{survey_year}/{str(survey_year + 1)[-2:]}"
                # Annual forecasting uses the previous wave; monthly forecasting
                # uses the same month one year earlier.
                lag_values = history.get(lag_key)
                if lag_values is None and frequency == "monthly":
                    # At the forecast origin a few low-n cells have no observed
                    # value in one month. Prefer the most recent observed value
                    # for the same calendar month; otherwise use the latest
                    # available value for that borough/activity group.
                    lag_date = pd.Timestamp(lag_key[2])
                    same_month = [
                        (key[2], value) for key, value in history.items()
                        if key[0] == borough and key[1] == level
                        and isinstance(key[2], pd.Timestamp)
                        and key[2].month == lag_date.month
                        and key[2] <= lag_date
                    ]
                    candidates = same_month or [
                        (key[2], value) for key, value in history.items()
                        if key[0] == borough and key[1] == level
                        and isinstance(key[2], pd.Timestamp)
                        and key[2] <= lag_date
                    ]
                    if candidates:
                        lag_values = max(candidates, key=lambda x: x[0])[1]
                if lag_values is None:
                    raise KeyError(f"Missing recursive lag {lag_key}")
                row = {
                    "borough_id": borough,
                    "borough_name": g.borough_name,
                    "borough_code": g.borough_code,
                    "inner_outer_code": int(g.inner_outer_code),
                    "inner_outer": g.inner_outer,
                    "activity_level_code": level,
                    "activity_level": g.activity_level,
                    "borough_cat": str(borough),
                    "activity_cat": str(level),
                    "inner_outer_cat": str(int(g.inner_outer_code)),
                    "survey_year": survey_year,
                    "survey_wave": survey_wave,
                    "period_start": period_start,
                    "scenario": scenario,
                }
                row.update(dict(zip(LAG_COLS, lag_values)))
                rows.append(future_covid_features(row, frequency, scenario))
            step = pd.DataFrame(rows)
            # Predict all groups for this period.  Naive simply copies the lag;
            # fitted models use the validation-selected, all-six-years refit.
            if selection.chosen_name == "Naive":
                pred = step[list(LAG_COLS)].to_numpy()
            else:
                pred = predict_pipeline(selection.final_model, step, frequency)
            for i, state_col in enumerate(STATE_COLS):
                step[state_col] = pred[:, i]
            # Save current predictions so later recursive steps can use them.
            for _, r in step.iterrows():
                key_period = int(period) if frequency == "annual" else pd.Timestamp(period)
                history[(int(r.borough_id), int(r.activity_level_code), key_period)] = r[list(STATE_COLS)].to_numpy(float)
            outputs.append(step)
    # Derive the same overlapping rates and exposure shares used for observations.
    forecast = pd.concat(outputs, ignore_index=True)
    forecast["indoor_rate"] = forecast["indoor_only_rate"] + forecast["both_rate"]
    forecast["outdoor_rate"] = forecast["outdoor_only_rate"] + forecast["both_rate"]
    denom = forecast["indoor_only_rate"] + forecast["outdoor_only_rate"] + 2 * forecast["both_rate"]
    forecast["indoor_exposure_share"] = (forecast["indoor_only_rate"] + forecast["both_rate"]) / denom.replace(0, np.nan)
    forecast["outdoor_exposure_share"] = 1 - forecast["indoor_exposure_share"]
    return forecast


In [15]:
# =============================================================================
# STEP 14 — Roll borough forecasts up for London-wide plots and summaries
# =============================================================================
def weighted_rollup(
    rates: pd.DataFrame,
    group_cols: list[str],
    reference_weights: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Aggregate compositions while preserving their population weighting.

    Observed panels use their own survey weight sums.  Future panels use a fixed
    2022 reference distribution so that a changing aggregation denominator does
    not create artificial future trends.
    """
    work = rates.copy()
    if reference_weights is not None:
        keys = ["borough_id", "activity_level_code"]
        work = work.merge(reference_weights[keys + ["aggregation_weight"]], on=keys, how="left")
    elif "weight_sum" in work:
        work["aggregation_weight"] = work["weight_sum"]
    else:
        raise ValueError("Weights are required for roll-up")
    for col in STATE_COLS:
        work[f"_w_{col}"] = work["aggregation_weight"] * work[col]
    g = work.groupby(group_cols, observed=True, dropna=False)
    out = g["aggregation_weight"].sum().reset_index(name="aggregation_weight")
    for col in STATE_COLS:
        num = g[f"_w_{col}"].sum().reset_index(name="num")
        out = out.merge(num, on=group_cols, how="left")
        out[col] = out.pop("num") / out["aggregation_weight"]
    out["indoor_rate"] = out["indoor_only_rate"] + out["both_rate"]
    out["outdoor_rate"] = out["outdoor_only_rate"] + out["both_rate"]
    denom = out["indoor_only_rate"] + out["outdoor_only_rate"] + 2 * out["both_rate"]
    out["indoor_exposure_share"] = (out["indoor_only_rate"] + out["both_rate"]) / denom.replace(0, np.nan)
    out["outdoor_exposure_share"] = 1 - out["indoor_exposure_share"]
    return out


In [16]:
# =============================================================================
# STEP 15 — Formally compare indoor/outdoor preference by activity level
# =============================================================================
# This section completes the second half of Question 3.  Forecasting separate
# lines for three groups is descriptive; the functions below additionally
# quantify group differences and their uncertainty in the observed data.

def benjamini_hochberg(p_values: pd.Series) -> np.ndarray:
    """Adjust a family of p-values while controlling the false discovery rate.

    The same three pairwise comparisons are repeated for London and 32
    boroughs.  Benjamini-Hochberg correction avoids declaring many differences
    significant merely because a large number of tests were performed.
    """
    p = pd.to_numeric(p_values, errors="coerce").to_numpy(float)
    adjusted = np.full(len(p), np.nan)
    valid = np.flatnonzero(np.isfinite(p))
    if not len(valid):
        return adjusted
    order = valid[np.argsort(p[valid])]
    ranked = p[order]
    m = len(ranked)
    corrected = ranked * m / np.arange(1, m + 1)
    corrected = np.minimum.accumulate(corrected[::-1])[::-1]
    adjusted[order] = np.clip(corrected, 0, 1)
    return adjusted


def observed_preference_bootstrap(
    respondents: pd.DataFrame,
    repetitions: int = 500,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """Estimate pairwise activity-level preference differences with 95% CIs.

    Indoor preference is defined as indoor exposure divided by all recorded
    indoor/outdoor exposure.  Thus indoor-only contributes 1 indoor exposure,
    outdoor-only contributes 1 outdoor exposure, and 'both' contributes one of
    each.  Respondents with neither recorded location have a zero denominator
    and are excluded from this *preference* comparison, but remain in all
    participation-rate models elsewhere in the script.

    A Bayesian bootstrap multiplies each respondent's survey weight by an
    independent Exp(1) draw.  It is computationally efficient for weighted
    survey estimates and gives a transparent empirical uncertainty interval.
    The pooled six-wave analysis gives every survey wave equal total weight;
    the latest-wave analysis uses the original 2022/23 annual weights.
    """
    if repetitions < 100:
        raise ValueError("--bootstrap-reps must be at least 100")
    work = respondents.copy()
    work["exposure_count"] = work["any_indoor"] + work["any_outdoor"]
    work = work[
        work["borough_id"].notna()
        & work["activity_level_code"].isin(ACTIVITY_LABELS)
        & pd.to_numeric(work["wt_final"], errors="coerce").gt(0)
        & work["exposure_count"].gt(0)
    ].copy()
    work["wt_final"] = pd.to_numeric(work["wt_final"], errors="coerce")
    rng = np.random.default_rng(random_state)
    records: list[dict[str, Any]] = []

    scopes = (
        ("all_six_waves_equal_wave_weight", work),
        ("latest_wave_2022_23", work[work["survey_year"] == 2022]),
    )
    geographies = [("London", "London", "London", None)] + [
        ("Borough", name, code, borough_id)
        for borough_id, (name, code) in BOROUGH_LOOKUP.items()
    ]
    for scope_name, scope in scopes:
        for geography_level, geography_name, geography_code, borough_id in geographies:
            sample = scope if borough_id is None else scope[scope["borough_id"] == borough_id]
            if sample.empty:
                continue
            weights = sample["wt_final"].to_numpy(float)
            # Equalise total weight across waves for the pooled estimand so a
            # large survey wave cannot dominate the six-year conclusion.
            if scope_name.startswith("all_six"):
                wave_total = sample.groupby("survey_year", observed=True)["wt_final"].transform("sum")
                weights = weights / wave_total.to_numpy(float)
            levels = sample["activity_level_code"].to_numpy(int)
            indoor = sample["any_indoor"].to_numpy(float)
            exposure = sample["exposure_count"].to_numpy(float)
            point_num = np.bincount(levels, weights=weights * indoor, minlength=3)
            point_den = np.bincount(levels, weights=weights * exposure, minlength=3)
            point = np.divide(
                point_num, point_den,
                out=np.full(3, np.nan), where=point_den > 0,
            )
            raw_n = np.bincount(levels, minlength=3)
            weight_sq = np.bincount(levels, weights=(weights * exposure) ** 2, minlength=3)
            effective_n = np.divide(
                point_den ** 2, weight_sq,
                out=np.full(3, np.nan), where=weight_sq > 0,
            )
            draws = np.full((repetitions, 3), np.nan, dtype=float)
            for b in range(repetitions):
                boot_w = weights * rng.exponential(scale=1.0, size=len(sample))
                num = np.bincount(levels, weights=boot_w * indoor, minlength=3)
                den = np.bincount(levels, weights=boot_w * exposure, minlength=3)
                draws[b] = np.divide(num, den, out=np.full(3, np.nan), where=den > 0)
            for level_a, level_b, contrast_name in ACTIVITY_CONTRASTS:
                delta = draws[:, level_a] - draws[:, level_b]
                finite_delta = delta[np.isfinite(delta)]
                estimable = bool(np.isfinite(point[level_a]) and np.isfinite(point[level_b]) and len(finite_delta))
                if estimable:
                    n_nonpositive = int(np.count_nonzero(finite_delta <= 0))
                    n_nonnegative = int(np.count_nonzero(finite_delta >= 0))
                    p_value = min(1.0, 2 * (min(n_nonpositive, n_nonnegative) + 1) / (len(finite_delta) + 1))
                    ci_low = float(np.quantile(finite_delta, 0.025))
                    ci_high = float(np.quantile(finite_delta, 0.975))
                else:
                    p_value = np.nan
                    ci_low = np.nan
                    ci_high = np.nan
                records.append({
                    "scope": scope_name,
                    "geography_level": geography_level,
                    "geography_name": geography_name,
                    "geography_code": geography_code,
                    "group_a_code": level_a,
                    "group_a": ACTIVITY_LABELS[level_a],
                    "group_b_code": level_b,
                    "group_b": ACTIVITY_LABELS[level_b],
                    "contrast": contrast_name,
                    "group_a_indoor_exposure_share": point[level_a],
                    "group_b_indoor_exposure_share": point[level_b],
                    "difference_a_minus_b": point[level_a] - point[level_b],
                    "ci_95_low": ci_low,
                    "ci_95_high": ci_high,
                    "bootstrap_p_two_sided": p_value,
                    "bootstrap_repetitions": repetitions,
                    "exposed_raw_n": int(len(sample)),
                    "group_a_exposed_raw_n": int(raw_n[level_a]),
                    "group_b_exposed_raw_n": int(raw_n[level_b]),
                    "group_a_effective_n": float(effective_n[level_a]),
                    "group_b_effective_n": float(effective_n[level_b]),
                    "minimum_group_exposed_raw_n": int(min(raw_n[level_a], raw_n[level_b])),
                    "minimum_group_effective_n": float(min(effective_n[level_a], effective_n[level_b])),
                    "low_group_raw_n_below_20": bool(min(raw_n[level_a], raw_n[level_b]) < 20),
                    "low_group_effective_n_below_20": bool(min(effective_n[level_a], effective_n[level_b]) < 20),
                    "estimable": estimable,
                })
    result = pd.DataFrame(records)
    # Correct within each analysis scope, because the pooled and latest-wave
    # results answer distinct questions and should be reported separately.
    result["p_fdr_bh"] = result.groupby("scope", group_keys=False)["bootstrap_p_two_sided"].transform(
        lambda x: benjamini_hochberg(x)
    )
    result["significant_95_ci"] = result["estimable"] & ((result["ci_95_low"] > 0) | (result["ci_95_high"] < 0))
    result["significant_fdr_05"] = result["estimable"] & (result["p_fdr_bh"] < 0.05)
    return result


def _wls_fit(x: np.ndarray, y: np.ndarray, weights: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Fit WLS and return coefficients with an HC1 sandwich covariance matrix."""
    w = np.asarray(weights, float)
    w = w / np.nanmean(w)
    xtwx_inv = np.linalg.pinv(x.T @ (w[:, None] * x))
    beta = xtwx_inv @ (x.T @ (w * y))
    residual = y - x @ beta
    score = x * (w * residual)[:, None]
    meat = score.T @ score
    n, k = x.shape
    hc1 = n / max(n - k, 1)
    covariance = hc1 * xtwx_inv @ meat @ xtwx_inv
    return beta, covariance


def _joint_wald(beta: np.ndarray, covariance: np.ndarray, indices: list[int]) -> tuple[float, int, float]:
    """Calculate a chi-square Wald test that selected coefficients equal zero."""
    b = beta[indices]
    v = covariance[np.ix_(indices, indices)]
    statistic = float(b.T @ np.linalg.pinv(v) @ b)
    degrees = int(np.linalg.matrix_rank(v))
    return statistic, degrees, float(chi2.sf(statistic, degrees))


def preference_global_tests(respondents: pd.DataFrame) -> pd.DataFrame:
    """Run omnibus weighted tests for activity level and borough interaction.

    The response is 1 for an indoor-only exposure, 0 for an outdoor-only
    exposure and 0.5 for a respondent reporting both; a 'both' respondent gets
    twice the exposure weight.  This makes the fitted mean exactly consistent
    with indoor_exposure_share.  Year fixed effects control ordinary wave-level
    change.  The reduced model tests an adjusted overall activity-level effect;
    the full model tests whether that effect varies across boroughs.
    """
    work = respondents.copy()
    work["exposure_count"] = work["any_indoor"] + work["any_outdoor"]
    work = work[
        work["borough_id"].notna()
        & work["activity_level_code"].isin(ACTIVITY_LABELS)
        & pd.to_numeric(work["wt_final"], errors="coerce").gt(0)
        & work["exposure_count"].gt(0)
    ].copy()
    y = work["any_indoor"].to_numpy(float) / work["exposure_count"].to_numpy(float)
    weights = work["wt_final"].to_numpy(float) * work["exposure_count"].to_numpy(float)
    year = pd.get_dummies(work["survey_year"].astype(int), prefix="year", drop_first=True, dtype=float)
    borough = pd.get_dummies(work["borough_id"].astype(int), prefix="borough", drop_first=True, dtype=float)
    activity = pd.get_dummies(work["activity_level_code"].astype(int), prefix="activity", drop_first=True, dtype=float)
    intercept = pd.DataFrame({"intercept": np.ones(len(work))}, index=work.index)

    reduced = pd.concat([intercept, year, borough, activity], axis=1)
    beta_r, cov_r = _wls_fit(reduced.to_numpy(float), y, weights)
    activity_idx = [reduced.columns.get_loc(c) for c in activity.columns]
    stat, degrees, p_value = _joint_wald(beta_r, cov_r, activity_idx)
    rows = [{
        "test": "Adjusted overall activity-level effect",
        "null_hypothesis": "Indoor exposure share is equal across activity levels after year and borough adjustment",
        "wald_chi_square": stat,
        "degrees_of_freedom": degrees,
        "p_value": p_value,
        "raw_n_with_recorded_exposure": len(work),
    }]

    interaction_parts = []
    for b_col in borough.columns:
        for a_col in activity.columns:
            interaction_parts.append((borough[b_col] * activity[a_col]).rename(f"{b_col}:{a_col}"))
    interactions = pd.concat(interaction_parts, axis=1)
    full = pd.concat([reduced, interactions], axis=1)
    beta_f, cov_f = _wls_fit(full.to_numpy(float), y, weights)
    interaction_idx = [full.columns.get_loc(c) for c in interactions.columns]
    stat, degrees, p_value = _joint_wald(beta_f, cov_f, interaction_idx)
    rows.append({
        "test": "Borough by activity-level interaction",
        "null_hypothesis": "Activity-level differences in indoor exposure share do not vary across boroughs",
        "wald_chi_square": stat,
        "degrees_of_freedom": degrees,
        "p_value": p_value,
        "raw_n_with_recorded_exposure": len(work),
    })
    return pd.DataFrame(rows)


def pairwise_forecast_contrasts(
    rates: pd.DataFrame,
    group_cols: list[str],
    geography_level: str,
) -> pd.DataFrame:
    """Convert three activity-level forecast rows into explicit pairwise gaps."""
    metrics = ["indoor_exposure_share", "indoor_rate", "outdoor_rate"]
    records: list[dict[str, Any]] = []
    for keys, part in rates.groupby(group_cols, observed=True, dropna=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        by_level = part.set_index("activity_level_code")
        if not set(ACTIVITY_LABELS).issubset(by_level.index):
            continue
        base = dict(zip(group_cols, keys))
        for level_a, level_b, contrast_name in ACTIVITY_CONTRASTS:
            row = {
                **base,
                "geography_level": geography_level,
                "group_a_code": level_a,
                "group_a": ACTIVITY_LABELS[level_a],
                "group_b_code": level_b,
                "group_b": ACTIVITY_LABELS[level_b],
                "contrast": contrast_name,
            }
            for metric in metrics:
                row[f"{metric}_difference_a_minus_b"] = float(by_level.loc[level_a, metric] - by_level.loc[level_b, metric])
            records.append(row)
    return pd.DataFrame(records)


def save_question3_summaries(
    annual: pd.DataFrame,
    annual_fc: pd.DataFrame,
    respondents: pd.DataFrame,
    output_dir: Path,
    bootstrap_reps: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Save London totals, activity-level contrasts, CIs and omnibus tests."""
    ref = annual.loc[
        annual.survey_year == 2022,
        ["borough_id", "activity_level_code", "weight_sum"],
    ].rename(columns={"weight_sum": "aggregation_weight"})
    london_observed = weighted_rollup(annual, ["survey_year", "survey_wave", "activity_level_code", "activity_level"])
    london_overall_observed = weighted_rollup(annual, ["survey_year", "survey_wave"])
    london_forecast = weighted_rollup(
        annual_fc,
        ["survey_year", "survey_wave", "scenario", "activity_level_code", "activity_level"],
        ref,
    )
    london_overall_forecast = weighted_rollup(
        annual_fc, ["survey_year", "survey_wave", "scenario"], ref
    )
    london_observed.to_csv(output_dir / "q3_london_activitylevel_annual.csv", index=False)
    london_overall_observed.to_csv(output_dir / "q3_london_overall_annual.csv", index=False)
    london_forecast.to_csv(output_dir / "forecast_london_activitylevel_annual.csv", index=False)
    london_overall_forecast.to_csv(output_dir / "forecast_london_overall_annual.csv", index=False)

    observed_contrasts = observed_preference_bootstrap(respondents, bootstrap_reps)
    observed_contrasts.to_csv(output_dir / "activitylevel_observed_contrasts_bootstrap.csv", index=False)
    global_tests = preference_global_tests(respondents)
    global_tests.to_csv(output_dir / "activitylevel_global_tests.csv", index=False)

    london_fc_contrasts = pairwise_forecast_contrasts(
        london_forecast,
        ["survey_year", "survey_wave", "scenario"],
        "London",
    )
    borough_fc_contrasts = pairwise_forecast_contrasts(
        annual_fc,
        ["survey_year", "survey_wave", "scenario", "borough_id", "borough_name", "borough_code"],
        "Borough",
    )
    forecast_contrasts = pd.concat([london_fc_contrasts, borough_fc_contrasts], ignore_index=True)
    forecast_contrasts.to_csv(output_dir / "activitylevel_forecast_contrasts_annual.csv", index=False)
    return observed_contrasts, forecast_contrasts


def alr_epsilon_sensitivity(
    features: pd.DataFrame,
    observed_annual: pd.DataFrame,
    selection: ModelSelection,
    frequency: str,
) -> pd.DataFrame:
    """Compare 10^-4, 10^-5 and 10^-6 without using test data to choose epsilon.

    The model family and hyperparameters are frozen at the primary rolling-CV
    choice.  Each epsilon is then evaluated on the same three validation folds
    and the untouched final test.  For the annual model, the table also reports
    the 2028/29 London indoor-exposure forecast under persistent COVID legacy.
    """
    clean = features.dropna(
        subset=list(LAG_COLS) + list(STATE_COLS) + ["effective_n"]
    ).copy()
    rows: list[dict[str, Any]] = []
    for epsilon in ALR_EPS_SENSITIVITY:
        fold_metrics = []
        for train_end, valid_year, fold_label in ROLLING_ORIGIN_FOLDS:
            train = clean[clean.survey_year.between(2017, train_end)]
            valid = clean[clean.survey_year == valid_year]
            y = valid[list(STATE_COLS)].to_numpy()
            if selection.chosen_name == "Naive":
                pred = valid[list(LAG_COLS)].to_numpy()
            else:
                model = fit_pipeline(
                    selection.chosen_name,
                    selection.chosen_params,
                    frequency,
                    train,
                    epsilon=epsilon,
                )
                pred = predict_pipeline(model, valid, frequency)
            fold_metrics.append(
                metric_row(y, pred, valid["effective_n"].to_numpy())
            )
        cv_tv = float(np.mean([x["weighted_tv"] for x in fold_metrics]))
        cv_mae = float(np.mean([x["weighted_mae"] for x in fold_metrics]))

        train_test = clean[clean.survey_year.between(2017, 2021)]
        test = clean[clean.survey_year == FINAL_TEST_YEAR]
        if selection.chosen_name == "Naive":
            test_pred = test[list(LAG_COLS)].to_numpy()
            fitted_all = None
        else:
            test_model = fit_pipeline(
                selection.chosen_name,
                selection.chosen_params,
                frequency,
                train_test,
                epsilon=epsilon,
            )
            test_pred = predict_pipeline(test_model, test, frequency)
            fitted_all = fit_pipeline(
                selection.chosen_name,
                selection.chosen_params,
                frequency,
                clean,
                epsilon=epsilon,
            )
        test_metric = metric_row(
            test[list(STATE_COLS)].to_numpy(),
            test_pred,
            test["effective_n"].to_numpy(),
        )
        forecast_2028 = np.nan
        if frequency == "annual":
            sensitivity_selection = ModelSelection(
                frequency,
                selection.chosen_name,
                selection.chosen_params,
                fitted_all,
                pd.DataFrame(),
                epsilon,
            )
            fc = recursive_forecast(features, sensitivity_selection, frequency)
            ref = observed_annual.loc[
                observed_annual.survey_year == 2022,
                ["borough_id", "activity_level_code", "weight_sum"],
            ].rename(columns={"weight_sum": "aggregation_weight"})
            london = weighted_rollup(
                fc[
                    (fc.survey_year == 2028)
                    & (fc.scenario == "persistent_legacy")
                ],
                ["survey_year", "scenario"],
                ref,
            )
            forecast_2028 = float(london["indoor_exposure_share"].iloc[0])
        rows.append({
            "frequency": frequency,
            "selected_model_frozen": selection.chosen_name,
            "selected_params_frozen": json.dumps(selection.chosen_params, sort_keys=True),
            "alr_epsilon": epsilon,
            "is_primary": epsilon == ALR_EPS,
            "rolling_cv_mean_weighted_tv": cv_tv,
            "rolling_cv_mean_weighted_mae": cv_mae,
            "final_test_weighted_tv_reporting_only": test_metric["weighted_tv"],
            "final_test_weighted_mae_reporting_only": test_metric["weighted_mae"],
            "forecast_2028_29_london_indoor_exposure_share": forecast_2028,
            "rationale": (
                "10^-6 is primary because it is the smallest tested replacement, "
                "minimising distortion while keeping ALR finite; alternatives quantify sensitivity"
            ),
        })
    return pd.DataFrame(rows)


def pooling_strength_sensitivity(
    panel: pd.DataFrame,
    frequency: str,
) -> pd.DataFrame:
    """Quantify how chosen partial-pooling strengths alter direct estimates."""
    time_cols = ["survey_year"] if frequency == "annual" else ["period_start"]
    default_strength = (
        ANNUAL_POOLING_STRENGTH if frequency == "annual"
        else MONTHLY_POOLING_STRENGTH
    )
    strengths = (
        (1.0, 5.0, 10.0) if frequency == "annual"
        else (10.0, 20.0, 40.0)
    )
    raw = panel.copy()
    for col in STATE_COLS:
        raw[col] = raw[f"{col}_raw"]
    raw["indoor_rate"] = raw["indoor_only_rate"] + raw["both_rate"]
    raw["outdoor_rate"] = raw["outdoor_only_rate"] + raw["both_rate"]
    raw_denom = (
        raw["indoor_only_rate"]
        + raw["outdoor_only_rate"]
        + 2 * raw["both_rate"]
    )
    raw["indoor_exposure_share"] = (
        raw["indoor_only_rate"] + raw["both_rate"]
    ) / raw_denom.replace(0, np.nan)
    raw["outdoor_exposure_share"] = 1 - raw["indoor_exposure_share"]
    pooled_by_strength = {
        strength: shrink_composition(raw, time_cols, strength)
        for strength in strengths
    }
    primary_pooled = pooled_by_strength[default_strength]
    results = []
    for strength, pooled in pooled_by_strength.items():
        component_change = np.abs(
            pooled[list(STATE_COLS)].to_numpy()
            - raw[list(STATE_COLS)].to_numpy()
        )
        exposure_change = np.abs(
            pooled["indoor_exposure_share"].to_numpy()
            - raw["indoor_exposure_share"].to_numpy()
        )
        component_vs_primary = np.abs(
            pooled[list(STATE_COLS)].to_numpy()
            - primary_pooled[list(STATE_COLS)].to_numpy()
        )
        exposure_vs_primary = np.abs(
            pooled["indoor_exposure_share"].to_numpy()
            - primary_pooled["indoor_exposure_share"].to_numpy()
        )
        results.append({
            "frequency": frequency,
            "prior_strength": strength,
            "is_primary": strength == default_strength,
            "mean_abs_component_change_from_direct": float(np.nanmean(component_change)),
            "max_abs_component_change_from_direct": float(np.nanmax(component_change)),
            "mean_abs_indoor_exposure_change_from_direct": float(np.nanmean(exposure_change)),
            "max_abs_indoor_exposure_change_from_direct": float(np.nanmax(exposure_change)),
            "mean_abs_component_difference_vs_primary": float(np.nanmean(component_vs_primary)),
            "max_abs_component_difference_vs_primary": float(np.nanmax(component_vs_primary)),
            "mean_abs_indoor_exposure_difference_vs_primary": float(np.nanmean(exposure_vs_primary)),
            "max_abs_indoor_exposure_difference_vs_primary": float(np.nanmax(exposure_vs_primary)),
            "rationale": (
                "Primary strength is comparable to 5 annual or 20 monthly effective observations; "
                "weaker/stronger alternatives measure threshold sensitivity"
            ),
        })
    return pd.DataFrame(results)


def save_covid_impact_summary(
    annual: pd.DataFrame,
    annual_fc: pd.DataFrame,
    output_dir: Path,
) -> None:
    """Save observed pre/during/post-COVID changes and future scenario effects."""
    observed = weighted_rollup(annual, ["survey_year", "survey_wave"])
    observed["period"] = np.select(
        [observed.survey_year <= 2019, observed.survey_year == 2020],
        ["pre_COVID_2017_18_to_2019_20", "COVID_disruption_2020_21"],
        default="post_disruption_2021_22_to_2022_23",
    )
    period_means = observed.groupby("period", as_index=False)[
        ["indoor_rate", "outdoor_rate", "indoor_exposure_share"]
    ].mean()
    baseline = period_means[
        period_means.period == "pre_COVID_2017_18_to_2019_20"
    ].iloc[0]
    for metric in ("indoor_rate", "outdoor_rate", "indoor_exposure_share"):
        period_means[f"{metric}_change_vs_pre_COVID"] = period_means[metric] - baseline[metric]
    period_means["analysis_type"] = "observed_period_comparison"

    ref = annual.loc[
        annual.survey_year == 2022,
        ["borough_id", "activity_level_code", "weight_sum"],
    ].rename(columns={"weight_sum": "aggregation_weight"})
    future = weighted_rollup(
        annual_fc,
        ["survey_year", "survey_wave", "scenario"],
        ref,
    )
    wide = future.pivot(
        index=["survey_year", "survey_wave"],
        columns="scenario",
        values=["indoor_rate", "outdoor_rate", "indoor_exposure_share"],
    )
    wide.columns = [f"{metric}__{scenario}" for metric, scenario in wide.columns]
    wide = wide.reset_index()
    for metric in ("indoor_rate", "outdoor_rate", "indoor_exposure_share"):
        wide[f"{metric}_persistent_minus_recovery"] = (
            wide[f"{metric}__persistent_legacy"]
            - wide[f"{metric}__legacy_recovery"]
        )
    wide["analysis_type"] = "future_COVID_scenario_sensitivity"
    observed.to_csv(output_dir / "covid_observed_annual_series.csv", index=False)
    period_means.to_csv(output_dir / "covid_observed_period_comparison.csv", index=False)
    wide.to_csv(output_dir / "covid_forecast_scenario_comparison.csv", index=False)


def save_algorithm_specifications(output_dir: Path) -> None:
    """Write a concise machine-readable explanation of every candidate model."""
    rows = [
        {
            "algorithm": "Naive persistence",
            "definition": "Copies the previous survey wave (annual) or same month one year earlier (monthly)",
            "candidate_parameters": "none",
        },
        {
            "algorithm": "Ridge regression",
            "definition": "Linear multi-output regression in ALR space with L2 coefficient shrinkage",
            "candidate_parameters": "alpha in {0.1, 1, 10, 100}",
        },
        {
            "algorithm": "Random Forest",
            "definition": "Average of bootstrap regression trees fitted jointly to the three ALR coordinates",
            "candidate_parameters": "250 trees; (max_depth=8, min_leaf=5) or (unlimited, min_leaf=10)",
        },
        {
            "algorithm": "Gradient Boosting",
            "definition": "Sequential Huber-loss regression trees, one ensemble per ALR coordinate",
            "candidate_parameters": "120/180 trees; learning_rate 0.03/0.05; max_depth 2/3; min_leaf 5/10",
        },
    ]
    pd.DataFrame(rows).to_csv(output_dir / "algorithm_specifications.csv", index=False)


In [17]:
# =============================================================================
# STEP 16 — Create publication-ready visualisations
# =============================================================================
def make_plots(
    annual: pd.DataFrame,
    monthly: pd.DataFrame,
    annual_fc: pd.DataFrame,
    monthly_fc: pd.DataFrame,
    metrics: pd.DataFrame,
    observed_contrasts: pd.DataFrame,
    forecast_contrasts: pd.DataFrame,
    output_dir: Path,
) -> None:
    """Create model, forecast and activity-level difference figures."""
    # Use one consistent visual theme and keep figures in a separate folder.
    sns.set_theme(style="whitegrid", context="talk")
    plots = output_dir / "plots"
    plots.mkdir(exist_ok=True)
    # Fix future aggregation weights at the last observed wave (2022/23).
    ref = annual.loc[annual.survey_year == 2022, ["borough_id", "activity_level_code", "weight_sum"]].rename(columns={"weight_sum": "aggregation_weight"})

    # Plot 1: London-wide observed history plus six-year annual forecasts.
    actual_london = weighted_rollup(annual, ["survey_year"])
    fc_london = weighted_rollup(annual_fc, ["survey_year", "scenario"], ref)
    fig, ax = plt.subplots(figsize=(11, 6))
    for col, label, color in [("indoor_rate", "Indoor", "#276FBF"), ("outdoor_rate", "Outdoor", "#2A9D8F")]:
        ax.plot(actual_london.survey_year, actual_london[col], marker="o", color=color, label=f"{label}: observed")
        base = fc_london[fc_london.scenario == "persistent_legacy"]
        recovery = fc_london[fc_london.scenario == "legacy_recovery"]
        ax.plot(base.survey_year, base[col], marker="o", linestyle="--", color=color, label=f"{label}: forecast")
        low = np.minimum(base[col].to_numpy(), recovery[col].to_numpy())
        high = np.maximum(base[col].to_numpy(), recovery[col].to_numpy())
        ax.fill_between(base.survey_year, low, high, color=color, alpha=.15)
    ax.axvline(2019.5, color="#D1495B", linestyle=":", label="COVID onset")
    ax.set(title="London indoor/outdoor participation: observed and forecast", xlabel="Survey-wave start year", ylabel="Weighted population rate", ylim=(0, None))
    ax.legend(ncol=2, fontsize=10)
    fig.tight_layout(); fig.savefig(plots / "01_london_annual_forecast.png", dpi=180); plt.close(fig)

    # Plot 2: all four models on both validation and untouched test data.
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=False)
    for ax, frequency in zip(axes, ("annual", "monthly")):
        cv_shown = metrics[
            (metrics.frequency == frequency)
            & (metrics.split == "rolling_validation_mean")
        ].sort_values(
            ["model", "selection_mean_rank", "weighted_tv", "weighted_mae"]
        ).groupby("model", as_index=False).head(1)
        test_shown = metrics[
            (metrics.frequency == frequency)
            & (metrics.split == "final_test_2022_23")
        ]
        shown_metrics = pd.concat([cv_shown, test_shown], ignore_index=True)
        shown_metrics["evaluation"] = shown_metrics["split"].map({
            "rolling_validation_mean": "Mean of 3 validation folds",
            "final_test_2022_23": "Untouched 2022/23 test",
        })
        sns.barplot(
            data=shown_metrics, x="model", y="weighted_tv", hue="evaluation",
            ax=ax, palette="Set2", errorbar=None,
        )
        ax.set(title=f"{frequency.title()} model comparison", xlabel="", ylabel="Weighted TV")
        ax.tick_params(axis="x", rotation=20, labelsize=10)
        ax.legend(title="Evaluation", fontsize=9)
    fig.suptitle("Model comparison (lower Total Variation is better)")
    fig.tight_layout(); fig.savefig(plots / "02_model_comparison.png", dpi=180); plt.close(fig)

    # Plot 3: borough comparison at the final forecast horizon.
    fc2028 = annual_fc[(annual_fc.survey_year == 2028) & (annual_fc.scenario == "persistent_legacy")]
    borough2028 = weighted_rollup(fc2028, ["borough_id", "borough_name", "borough_code"], ref)
    fig, ax = plt.subplots(figsize=(8, 10))
    shown = borough2028.sort_values("indoor_exposure_share")
    sns.barplot(data=shown, y="borough_name", x="indoor_exposure_share", orient="h", ax=ax, color="#5B8FF9")
    ax.set(title="2028/29 indoor exposure share by borough", xlabel="Indoor share of recorded location exposure", ylabel="Borough", xlim=(0, 1))
    fig.tight_layout(); fig.savefig(plots / "03_borough_2028_indoor_share.png", dpi=180); plt.close(fig)

    # Plot 4: compare the three overall physical-activity-level groups.
    by_level = weighted_rollup(annual_fc, ["survey_year", "scenario", "activity_level_code", "activity_level"], ref)
    base_level = by_level[by_level.scenario == "persistent_legacy"]
    fig, ax = plt.subplots(figsize=(11, 6))
    sns.lineplot(data=base_level, x="survey_year", y="indoor_exposure_share", hue="activity_level", marker="o", ax=ax)
    ax.set(title="Forecast indoor preference by activity level", xlabel="Survey-wave start year", ylabel="Indoor exposure share", ylim=(0, 1))
    fig.tight_layout(); fig.savefig(plots / "04_activity_level_forecast.png", dpi=180); plt.close(fig)

    # Plot 5: monthly seasonality and the sensitivity range between scenarios.
    monthly_ref = (
        monthly[monthly.survey_year == 2022]
        .groupby(["borough_id", "activity_level_code"], observed=True)["weight_sum"].mean()
        .reset_index(name="aggregation_weight")
    )
    actual_m = weighted_rollup(monthly, ["period_start"])
    fc_m = weighted_rollup(monthly_fc, ["period_start", "scenario"], monthly_ref)
    fig, ax = plt.subplots(figsize=(13, 6))
    ax.plot(actual_m.period_start, actual_m.indoor_exposure_share, color="#264653", label="Observed")
    base_m = fc_m[fc_m.scenario == "persistent_legacy"]
    rec_m = fc_m[fc_m.scenario == "legacy_recovery"]
    ax.plot(base_m.period_start, base_m.indoor_exposure_share, color="#E76F51", label="Forecast")
    base_values = base_m.indoor_exposure_share.to_numpy()
    recovery_values = rec_m.indoor_exposure_share.to_numpy()
    ax.fill_between(
        base_m.period_start,
        np.minimum(base_values, recovery_values),
        np.maximum(base_values, recovery_values),
        color="#E76F51", alpha=.18,
    )
    ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-07-01"), color="#D1495B", alpha=.1, label="COVID disruption window")
    ax.set(title="Monthly London indoor exposure share", xlabel="Calendar month", ylabel="Indoor exposure share", ylim=(0, 1))
    ax.legend(fontsize=10)
    fig.tight_layout(); fig.savefig(plots / "05_london_monthly_forecast.png", dpi=180); plt.close(fig)

    # Plot 6: formal latest-wave London contrasts with bootstrap uncertainty.
    latest_london = observed_contrasts[
        (observed_contrasts.scope == "latest_wave_2022_23")
        & (observed_contrasts.geography_level == "London")
    ].copy().sort_values("difference_a_minus_b")
    y_position = np.arange(len(latest_london))
    estimate = latest_london["difference_a_minus_b"].to_numpy()
    lower = latest_london["ci_95_low"].to_numpy()
    upper = latest_london["ci_95_high"].to_numpy()
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.errorbar(
        estimate, y_position,
        xerr=np.vstack([estimate - lower, upper - estimate]),
        fmt="o", color="#6A4C93", ecolor="#6A4C93", capsize=5,
    )
    ax.axvline(0, color="black", linestyle=":")
    ax.set_yticks(y_position, latest_london["contrast"], fontsize=12)
    ax.set(
        title="London activity-level differences in indoor preference (2022/23)",
        xlabel="Difference in indoor exposure share (95% bootstrap CI)",
        ylabel="",
    )
    ax.title.set_fontsize(16)
    ax.xaxis.label.set_fontsize(13)
    fig.tight_layout(); fig.savefig(plots / "06_activitylevel_differences_london_2022.png", dpi=180, bbox_inches="tight"); plt.close(fig)

    # Plot 7: borough heterogeneity in the final Active-vs-Inactive forecast.
    heat = forecast_contrasts[
        (forecast_contrasts.geography_level == "Borough")
        & (forecast_contrasts.survey_year == 2028)
        & (forecast_contrasts.scenario == "persistent_legacy")
        & (forecast_contrasts.contrast == "Active minus Inactive")
    ].copy()
    heat = heat.sort_values("indoor_exposure_share_difference_a_minus_b")
    fig, ax = plt.subplots(figsize=(10, 11))
    sns.barplot(
        data=heat, y="borough_name",
        x="indoor_exposure_share_difference_a_minus_b",
        color="#4C78A8", ax=ax,
    )
    ax.axvline(0, color="black", linestyle=":")
    ax.set(
        title="2028/29 forecast Active-minus-Inactive indoor-preference gap",
        xlabel="Difference in indoor exposure share",
        ylabel="Borough",
    )
    ax.title.set_fontsize(16)
    ax.xaxis.label.set_fontsize(13)
    ax.yaxis.label.set_fontsize(13)
    ax.tick_params(axis="y", labelsize=10)
    fig.tight_layout(); fig.savefig(plots / "07_borough_active_inactive_gap_2028.png", dpi=180, bbox_inches="tight"); plt.close(fig)


def write_audit(
    respondents: pd.DataFrame,
    activities: list[str],
    annual: pd.DataFrame,
    monthly: pd.DataFrame,
    selections: list[ModelSelection],
    output_dir: Path,
) -> None:
    """Save key row counts, checks, assumptions and selected models as JSON."""
    # This compact audit makes it easier to verify what a particular run used
    # without reopening every large CSV output.
    audit = {
        "respondent_rows": int(len(respondents)),
        "survey_year_counts": respondents.survey_year.value_counts().sort_index().astype(int).to_dict(),
        "borough_count": int(respondents.borough_id.nunique()),
        "comparable_location_activities": len(activities),
        "annual_panel_rows": int(len(annual)),
        "monthly_panel_rows": int(len(monthly)),
        "annual_low_effective_n_below_20": int(annual.low_effective_n_20.sum()),
        "monthly_low_effective_n_below_20": int(monthly.low_effective_n_20.sum()),
        "annual_composition_max_abs_error": float(np.abs(annual[list(STATE_COLS)].sum(axis=1) - 1).max()),
        "monthly_composition_max_abs_error": float(np.abs(monthly[list(STATE_COLS)].sum(axis=1) - 1).max()),
        "annual_partial_pooling_prior_strength": ANNUAL_POOLING_STRENGTH,
        "monthly_partial_pooling_prior_strength": MONTHLY_POOLING_STRENGTH,
        "alr_zero_replacement_epsilon": ALR_EPS,
        "alr_epsilon_sensitivity_values": list(ALR_EPS_SENSITIVITY),
        "rolling_origin_validation": [
            {
                "training": "2017/18--2018/19",
                "validation": "2019/20",
            },
            {
                "training": "2017/18--2019/20",
                "validation": "2020/21",
            },
            {
                "training": "2017/18--2020/21",
                "validation": "2021/22",
            },
        ],
        "final_test": "2022/23, used once after model and hyperparameter selection",
        "selection_rule": "lowest mean rank across 3-fold mean weighted TV and weighted MAE; TV then MAE are deterministic tie-breakers",
        "metric_rationale": {
            "TV": "evaluates the entire four-part probability composition as displaced mass",
            "MAE": "reports typical error in derived indoor/outdoor participation rates on the original scale and is interpretable in percentage points",
        },
        "selected_models": {s.frequency: {"name": s.chosen_name, "params": s.chosen_params} for s in selections},
        "important_definition": (
            "neither_recorded means neither an indoor nor outdoor location was recorded among qualifying comparable activities; "
            "it comprises no_qualifying_comparable_activity plus recent_activity_without_recorded_location and is not physical inactivity"
        ),
        "preference_definition": "indoor exposure / (indoor exposure + outdoor exposure); both contributes one exposure to each side",
        "missing_and_zero_rule": "CSV blanks and illegal codes become pandas NaN (not Python None); no imputation is used; valid zero codes remain zero and participate as negative/no indicators",
        "preference_inference": "Bayesian-bootstrap pairwise contrasts plus BH-FDR correction; low-n and non-estimable borough comparisons are retained and flagged",
        "covid_scenarios": {
            "persistent_legacy": "COVID-era level and time-since-COVID terms continue; no future disruption-window flag",
            "legacy_recovery": "future COVID-specific terms are set to zero; ordinary time trend and seasonality remain",
        },
    }
    (output_dir / "run_audit.json").write_text(json.dumps(audit, indent=2, ensure_ascii=False), encoding="utf-8")


def write_methods_readme(output_dir: Path) -> None:
    """Save an English methods note that can be reused in the dissertation."""
    text = """# Q3 revised workflow

## Chronological validation

The workflow uses three expanding-window rolling-origin validation folds:

1. train on 2017/18--2018/19 and validate on 2019/20;
2. train on 2017/18--2019/20 and validate on 2020/21;
3. train on 2017/18--2020/21 and validate on 2021/22.

Candidate models and hyperparameters are ranked using the mean weighted Total
Variation (TV) and mean weighted MAE across all three folds. TV measures the
probability mass assigned to the wrong parts of the four-category composition;
MAE gives the typical indoor/outdoor-rate error in percentage-point units. The model with
the lowest mean of the two metric ranks is selected. Only after selection is it
refitted through 2021/22 and evaluated once on the untouched 2022/23 test wave.

## Outcome definitions

The mutually exclusive outcome is: neither recorded, indoor only, outdoor only,
or both. "Neither recorded" does not mean physically inactive. It means that no
qualifying comparable activity had an indoor or outdoor location flag. The code
separates respondents with no qualifying activity from respondents who reported
a qualifying activity but had no recorded indoor/outdoor location.

Indoor exposure is `(indoor only + both) / (indoor only + outdoor only + 2*both)`.
It is an exposure-balance measure, not the proportion of people participating
indoors. A respondent in "both" contributes one indoor and one outdoor exposure.

## Missing values and valid zeroes

CSV blanks, non-numeric entries and codes outside each variable's legal set are
converted to pandas `NaN`; they are not converted to Python `None` and they are
not imputed. Legal zeroes remain zero. A zero in Months or Days therefore does
not satisfy the qualifying-participation rule, while zero in a location flag
means that location was not recorded for that activity.

## ALR and sensitivity analysis

The four probabilities are modelled through the additive log-ratio (ALR)
transform, using `both` as the reference component. Structural/sample zeroes are
replaced by epsilon before taking logarithms. The primary epsilon is 10^-6 to
minimise perturbation while keeping logarithms finite; 10^-4 and 10^-5 are
evaluated with the same model specification. Partial-pooling strengths are also
compared against weaker and stronger alternatives. Hyperparameters are treated
as candidate intensity parameters and are chosen only by rolling-origin CV.

## Forecast and COVID interpretation

The selected models are refitted on all observations from 2017/18--2022/23 and
produce recursive forecasts for every wave from 2023/24 through 2028/29. The
`persistent_legacy` and `legacy_recovery` scenarios quantify how future results
change when post-COVID controls continue or are set to zero. Their difference is
a scenario sensitivity range, not a causal estimate or confidence interval.
"""
    (output_dir / "METHODS_README.md").write_text(text, encoding="utf-8")


In [18]:
# =============================================================================
# STEP 17 — Run the complete workflow in the correct order
# =============================================================================
def main() -> None:
    """Execute data preparation, modelling, forecasting, plots and audit files."""
    # 17.1 Create the output folder before any result is written.
    args = parse_args()
    args.output_dir.mkdir(parents=True, exist_ok=True)

    # 17.2 Build the six-wave respondent dataset, then the annual/monthly panels.
    respondents, activities = make_respondent_data(args.input_dir, args.output_dir)
    annual, monthly = make_panels(respondents, args.output_dir)

    # 17.3 Add lagged/time/COVID-control predictors separately by frequency.
    annual_features = add_model_features(annual, "annual")
    monthly_features = add_model_features(monthly, "monthly")

    # 17.4 Compare Naive, Ridge, Random Forest and Gradient Boosting using the
    # three-fold rolling-origin design. Selection uses validation folds only.
    print("Selecting annual models ...", flush=True)
    annual_selection = select_and_test_models(annual_features, "annual")
    print("Selecting monthly models ...", flush=True)
    monthly_selection = select_and_test_models(monthly_features, "monthly")
    # Store both validation and untouched test results for honest reporting.
    metrics = pd.concat([annual_selection.metrics, monthly_selection.metrics], ignore_index=True)
    metrics.to_csv(args.output_dir / "model_metrics.csv", index=False)
    metrics[metrics.split == "rolling_validation_mean"].to_csv(
        args.output_dir / "model_selection_rolling_cv_summary.csv", index=False
    )

    # 17.5 Forecast six future years recursively using each frequency's selected
    # model refitted on all six observed waves.
    print("Forecasting 2023/24--2028/29 ...", flush=True)
    annual_fc = recursive_forecast(annual_features, annual_selection, "annual")
    monthly_fc = recursive_forecast(monthly_features, monthly_selection, "monthly")
    annual_fc.to_csv(args.output_dir / "forecast_annual_2023_2028.csv", index=False)
    monthly_fc.to_csv(args.output_dir / "forecast_monthly_2023_2028.csv", index=False)

    # 17.6 Compare all explicitly chosen numeric constants.  ALR epsilon is
    # evaluated at 10^-4, 10^-5 and 10^-6 with the selected model frozen;
    # partial-pooling strengths are compared with weaker/stronger alternatives.
    print("Running parameter sensitivity analyses ...", flush=True)
    epsilon_sensitivity = pd.concat([
        alr_epsilon_sensitivity(
            annual_features, annual, annual_selection, "annual"
        ),
        alr_epsilon_sensitivity(
            monthly_features, annual, monthly_selection, "monthly"
        ),
    ], ignore_index=True)
    epsilon_sensitivity.to_csv(
        args.output_dir / "alr_epsilon_sensitivity.csv", index=False
    )
    pooling_sensitivity = pd.concat([
        pooling_strength_sensitivity(annual, "annual"),
        pooling_strength_sensitivity(monthly, "monthly"),
    ], ignore_index=True)
    pooling_sensitivity.to_csv(
        args.output_dir / "partial_pooling_sensitivity.csv", index=False
    )
    save_covid_impact_summary(annual, annual_fc, args.output_dir)
    save_algorithm_specifications(args.output_dir)
    write_methods_readme(args.output_dir)

    # 17.7 Formally answer whether activity-level preferences differ. Observed
    # contrasts receive bootstrap CIs and adjusted p-values; future contrasts
    # are scenario forecasts and therefore are not assigned significance tests.
    print("Estimating activity-level differences ...", flush=True)
    observed_contrasts, forecast_contrasts = save_question3_summaries(
        annual, annual_fc, respondents, args.output_dir, args.bootstrap_reps,
    )

    # 17.8 Produce figures and a machine-readable audit of the completed run.
    make_plots(
        annual, monthly, annual_fc, monthly_fc, metrics,
        observed_contrasts, forecast_contrasts, args.output_dir,
    )
    write_audit(
        respondents, activities, annual, monthly,
        [annual_selection, monthly_selection], args.output_dir,
    )
    print(json.dumps({
        "annual_selected": annual_selection.chosen_name,
        "annual_params": annual_selection.chosen_params,
        "monthly_selected": monthly_selection.chosen_name,
        "monthly_params": monthly_selection.chosen_params,
        "output_dir": str(args.output_dir.resolve()),
    }, indent=2), flush=True)


# This guard runs main() only when the file is executed directly.  It prevents
# the entire analysis from running automatically when functions are imported by
# indoor_outdoor_forecasting.ipynb.
if __name__ == "__main__":
    main()


Reading 2017_data_179_activities.csv ...
Reading 2018_data_179_activities.csv ...
Reading 1920_london32_stable179.csv ...
Reading 2021_london32_stable179.csv ...
Reading year7_179activities.csv ...
Reading year8_179activities.csv ...
Selecting annual models ...
Selecting monthly models ...
Forecasting 2023/24--2028/29 ...
Running parameter sensitivity analyses ...
Estimating activity-level differences ...
{
  "annual_selected": "Random Forest",
  "annual_params": {
    "max_depth": 8,
    "min_samples_leaf": 5,
    "n_estimators": 250
  },
  "monthly_selected": "Random Forest",
  "monthly_params": {
    "max_depth": null,
    "min_samples_leaf": 10,
    "n_estimators": 250
  },
  "output_dir": "E:\\Dissertation 2026\\Siyan Xin\\2017~2022\\q3_outputs"
}
